# Session1_Task4_short — Product Performance Analysis

In [1]:
import pandas as pd, matplotlib.pyplot as plt, matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages

s = pd.read_csv('sales_transactions_cleaned.csv')
p = pd.read_csv('products.csv')
s['revenue'] = (s['quantity']*s['price']) - pd.to_numeric(s['discount_amount'],errors='coerce').fillna(0)

# clean price/cost: ลบ $ ออก
clean = lambda col: pd.to_numeric(col.astype(str).str.replace(r'[^0-9.\-]','',regex=True),errors='coerce').abs()
p['price_c'] = clean(p['price']); p['cost_c'] = clean(p['cost'])
p['category'] = p['category'].replace({'Pastry':'Pastries'})  # แก้ typo
p['profit_margin'] = p['price_c'] - p['cost_c']               # กำไรต่อชิ้น

# merge sales กับ products
tq = s.merge(p[['product_id','product_name','category','profit_margin']], on='product_id', how='left')

# revenue by category
cr = tq[tq['category'].isin(['Pastries','Bread','Tarte'])].groupby('category')['revenue'].sum().reindex(['Pastries','Bread','Tarte'])

# top 3 by quantity
t3 = (tq.groupby('product_name')[['quantity','revenue']].sum().nlargest(3,'quantity').reset_index()
       .rename(columns={'product_name':'Product Name','quantity':'Total Qty','revenue':'Total Revenue'})
       .assign(**{'Total Revenue': lambda x: x['Total Revenue'].apply(lambda v: f'${v:,.2f}')}))

def style(tbl, c):
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.5, 2.2)
    for (r,_),cell in tbl.get_celld().items():
        if r==0: cell.set_facecolor(c); cell.set_text_props(color='white',fontweight='bold')
        cell.set_edgecolor('lightgray')

with PdfPages('Session1_ProductPerformance_short.pdf') as pdf:
    # bar chart
    fig,ax = plt.subplots(figsize=(8,5))
    cr.plot(kind='bar',color=['tomato','steelblue','seagreen'],ax=ax,rot=0)
    ax.set(title='Total Revenue by Product Category',ylabel='Total Revenue ($)',xlabel='Category')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
    ax.grid(axis='y',linestyle='--',alpha=0.5); fig.tight_layout()
    pdf.savefig(fig,bbox_inches='tight'); plt.close()
    # table
    fig,ax = plt.subplots(figsize=(9,2.5)); ax.axis('off')
    ax.set_title('Top 3 Best-Selling Products',fontsize=13,fontweight='bold',pad=16)
    style(ax.table(cellText=t3.values,colLabels=t3.columns,loc='center',cellLoc='center'),'tomato')
    fig.tight_layout(); pdf.savefig(fig,bbox_inches='tight'); plt.close()

print('✅ Saved Session1_ProductPerformance_short.pdf')
# จุดสังเกต: PDF 2 หน้า, bar chart 3 สี, top3 มี 3 แถว

✅ Saved Session1_ProductPerformance_short.pdf
